# Regression fine-tuning on a high-offset target

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/f-inverse/jammi-ai/blob/py-v0.55.0/cookbook/notebooks/book/finetune-regression/finetune-regression.ipynb)

Built from [`cookbook/book/chapters/finetune-regression/finetune-regression.qmd`](https://github.com/f-inverse/jammi-ai/blob/main/cookbook/book/chapters/finetune-regression/finetune-regression.qmd). Run the setup
cell first; every other cell runs top to bottom.

In [ ]:
# Setup: jammi 0.55.0 — the CUDA engine on an sm_80+ GPU (L4, A100, …), the
# CPU engine otherwise — and the cookbook's library and fixtures, from the release's
# tag on GitHub. The chapter runs
# at `small` scale, over the committed samples with a compact text encoder: seconds
# to minutes, or for a chapter that fine-tunes an encoder, minutes on a GPU and up
# to hours on a CPU. The models it runs are downloaded from the Hugging Face Hub on
# first use. SCALE = "full" runs it over the published data and a larger text
# encoder instead, on the GPU: the chapters that fine-tune take hours there.
import os
import subprocess
import sys


def compute_capability() -> float:
    try:
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"],
            capture_output=True, text=True, check=True,
        ).stdout.split()
    except (OSError, subprocess.CalledProcessError):
        return 0.0
    return float(out[0]) if out else 0.0


gpu = compute_capability() >= 8.0
engine = "jammi-ai-native-cu12" if gpu else "jammi-ai-native"
server = "jammi-server-cu12" if gpu else "jammi-server"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jammi-ai==0.55.0", engine + "==0.55.0", "jammi-cookbook @ git+https://github.com/f-inverse/jammi-ai@py-v0.55.0#subdirectory=cookbook/book"], check=True)
SCALE = "small"
os.environ["JAMMI_COOKBOOK_SCALE"] = SCALE
print(f"engine: {engine}   scale: {SCALE}")

**Verbs:** `fine_tune(task="regression")` (the `regression_loss` objective) +
`infer(task="regression")` · **Background:** heteroscedastic regression — the
mean–variance Gaussian head (Nix & Weigend 1994), β-NLL (Seitzer et al. 2022), CRPS as a training
score (Gneiting & Raftery 2007), pinball / quantile regression (Koenker & Bassett 1978) · **Checked:**
measurement (held-out RMSE in years against the predict-the-mean baseline,
interval coverage).

A paper's publication **year** is a hard target for a regression head: it sits
near 2018 — a large *offset* — with a spread of only a few years. A head that
regressed the raw year would have to learn a bias of two thousand while its
variance branch sees residuals of a few units, and the optimisation degenerates.
The engine avoids that by construction: it trains the regression head against
the target **standardized** with the training data's own mean and standard
deviation, so the objective the optimizer sees is an `O(1)` residual whatever the
target's offset or scale, and it **de-standardizes** at serving, so predictions
come back in years. This chapter measures the consequence on three questions:

1. **Does the high-offset target fit, with a real served spread?**
2. **Does the fit beat the baseline every regression must beat** — predicting
   the held-out mean for every paper?
3. **How honest is each objective's band** — `beta_nll` (the default),
   `gaussian_nll`, `crps`, or `pinball`?

In [ ]:
import tempfile
from pathlib import Path

import jammi
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
from jammi_cookbook import datasets, encoders, scale
from jammi_cookbook.claims import claim

SCALE = scale.current()
# The fine-tune adapts the encoder's attention and trains the head with it, so the
# step is LoRA-sized: a head-sized step (5e-2) carries the loss past a finite value
# and the engine refuses the run.
BUDGET = {
    scale.Scale.SMALL: {"epochs": 10, "learning_rate": 3e-3},
    scale.Scale.FULL: {"epochs": 3, "learning_rate": 1e-3},
}[SCALE]
QUANTILES = [0.05, 0.5, 0.95]  # a 90% central interval and the median
db = jammi.connect(f"file://{tempfile.mkdtemp()}")
arxiv = datasets.arxiv(db, SCALE)

## The supervision and the held-out split

`fine_tune(task="regression")` reads a `(text, target)` schema: here a paper's
title and abstract, and its year. Every fifth paper (in key order) is held out;
the rest train. Nothing about the held-out papers reaches the fit.

In [ ]:
papers = db.sql(
    f"SELECT paper_id, title, abstract, year FROM {arxiv.papers}.public.{arxiv.papers} "
    "ORDER BY paper_id"
).to_pylist()
held_out = papers[::5]
training = [p for i, p in enumerate(papers) if i % 5]
work = Path(tempfile.mkdtemp())
pq.write_table(pa.table({
    "text": [f"{p['title']}. {p['abstract']}" for p in training],
    "target": [float(p["year"]) for p in training],
}), work / "years_train.parquet")
pq.write_table(pa.table({
    "paper_id": [p["paper_id"] for p in held_out],
    "text": [f"{p['title']}. {p['abstract']}" for p in held_out],
}), work / "years_test.parquet")
db.add_source("years_train", url=str(work / "years_train.parquet"), format="parquet")
db.add_source("years_test", url=str(work / "years_test.parquet"), format="parquet")
truth = {p["paper_id"]: float(p["year"]) for p in held_out}
spread = float(np.std(list(truth.values())))
print(f"train {len(training)} / held out {len(held_out)}; held-out year std {spread:.2f}")

## Each objective, fitted and measured

Each run is the same short LoRA fine-tune — the encoder's attention adapted, the
regression head trained with it — and only `regression_loss` changes. The
Gaussian objectives serve `predicted_mean` and `predicted_std`; `pinball` serves
one `quantile_{level}` column per requested level. Every number below is read
off the de-standardized serve output, in years.

In [ ]:
def fit(loss: str) -> pa.Table:
    knobs = {"quantile_levels": QUANTILES} if loss == "pinball" else {}
    job = db.fine_tune(
        source="years_train", base_model=encoders.text(SCALE), columns=["text", "target"],
        method="lora", task="regression", regression_loss=loss, batch_size=16,
        max_seq_length=128, backbone_dtype=encoders.training_dtype(SCALE), seed=0,
        **BUDGET, **knobs,
    )
    job.wait()
    return db.infer(
        source="years_test", model=job.output_model_id, columns=["text"],
        task="regression", key="paper_id",
    )


def measure(served: pa.Table, loss: str) -> dict:
    years = np.array([truth[k] for k in served.column("_row_id").to_pylist()])
    column = lambda name: np.asarray(served.column(name).to_pylist(), dtype=float)
    if loss == "pinball":
        low, point, high = (column(f"quantile_{q}") for q in QUANTILES)
    else:
        point, std = column("predicted_mean"), column("predicted_std")
        low, high = point - 1.6449 * std, point + 1.6449 * std
    return {
        "rmse": float(np.sqrt(np.mean((point - years) ** 2))),
        "coverage": float(np.mean((years >= low) & (years <= high))),
        "width": float(np.mean(high - low)),
        # The scale the band implies: its width over 2 × 1.645.
        "sigma": float(np.mean(high - low)) / (2 * 1.6449),
    }


results = {loss: measure(fit(loss), loss) for loss in ("beta_nll", "gaussian_nll", "crps", "pinball")}
print(f"{'objective':<14}{'rmse(y)':>9}{'σ(y)':>8}{'cov@90':>9}{'90% width(y)':>14}")
for loss, r in results.items():
    print(f"{loss:<14}{r['rmse']:>9.3f}{r['sigma']:>8.2f}{r['coverage']:>9.3f}{r['width']:>14.2f}")
print(f"{'(predict the mean)':<14}{spread:>9.3f}")

In [ ]:
gaussian = [results[loss] for loss in ("beta_nll", "gaussian_nll", "crps")]
claim("the Gaussian heads serve a spread, not a collapsed point",
      all(r["width"] > 0.1 * spread for r in gaussian))
claim("no objective beats predicting the held-out mean",
      all(r["rmse"] > 0.95 * spread for r in results.values()),
      f"baseline {spread:.3f} years")

## Reading the table

The last row is the baseline every regression must beat: predicting the
held-out mean for every paper has an RMSE equal to the held-out standard
deviation. Every objective lands on that line. The head fits — its predictions
come back in years, with a real spread — but a paper's publication year is
barely written in its title and abstract, so there is nothing for it to learn
beyond the mean. That is the first number to read after any regression
fine-tune: a model that does not beat the baseline has learned the target's
location, not the target.

What still separates the objectives is how honest each band is about that
error. A 90% interval should contain about 90% of the held-out years, and a
Gaussian band gets there when the scale it implies, its σ, is about as large as
the error the head makes; the table puts the two side by side. The target blurs
the comparison: a publication year is an integer, and with every centre near the
same value, coverage moves only when a band edge crosses a whole year. A band
whose σ sits a little below its RMSE can still cover, and the same objective on
more papers can fall short. The heads fit their scale on the training papers,
so nothing guarantees which side they land on for papers they have not seen.
Split-conformal calibration exists to make it a guarantee: calibrate the band's
width on held-out residuals (the conformal chapter's CQR does exactly this to a
served ±σ band).

The `pinball` head reaches its band differently. Every quantile level starts at
the same output, and the pinball loss moves each by a bounded, sign-driven
gradient — `q` or `1 − q` per row, whatever the size of the miss — so the levels
separate only as fast as the optimiser's steps carry them. A Gaussian head
needs only its mean and one scale to move.

In [ ]:
db.close()

## References

- Nix, David A., Weigend, Andreas S. (1994) *Estimating the Mean and Variance of the Target Probability Distribution* Proceedings of the 1994 IEEE International Conference on Neural Networks (ICNN'94) DOI 10.1109/ICNN.1994.374138.
- Seitzer, Maximilian, Tavakoli, Arash, Antic, Dimitrije, Martius, Georg (2022) *On the Pitfalls of Heteroscedastic Uncertainty Estimation with Probabilistic Neural Networks* International Conference on Learning Representations (ICLR) arXiv:2203.09168.
- Gneiting, Tilmann, Raftery, Adrian E. (2007) *Strictly Proper Scoring Rules, Prediction, and Estimation* Journal of the American Statistical Association.
- Koenker, Roger, Bassett, Gilbert (1978) *Regression Quantiles* Econometrica DOI 10.2307/1913643.